<a href="https://colab.research.google.com/github/smsag99/Thesis/blob/main/codes/Hierchical_Structure.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IMPORTANT :



In [1]:
path = '/content/drive/MyDrive/Thesis_Data'

In [6]:
import pandas as pd
df = pd.read_csv(path + '/Merged_Data')
df = df[df['Farm_Code'] != 1]
df[df['Farm_Code']==1]

,Farm_Code,Animal_ID,dtb,dtc,dtt,parity,milk_kg,fat_p,protein_p,cells,...,wind_speed_min,THI_1_avg,THI_1_min,THI_1_max,THI_2_avg,THI_2_min,THI_2_max,WHI_avg,WHI_min,WHI_max


## Claude Method

### Step 1 — Build the summing matrix S
#### This is the core of HTS. The S matrix maps bottom-level series (Animal × Parity) to every level above.


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(path + '/merged_data')
df = df[df['Farm_Code'] != 1]
df['dtt'] = pd.to_datetime(df['dtt'])

# Create a unique bottom-level key: Animal_Parity
df['series_key'] = df['Animal_ID'] + '_P' + df['parity'].astype(str)

# Pivot: each column = one bottom-level series, rows = dates
bottom = df.pivot_table(index='dtt', columns='series_key', values='milk_kg', aggfunc='sum').fillna(0)

# Animal-level aggregates (sum parities per animal per date)
animal_level = df.pivot_table(index='dtt', columns='Animal_ID', values='milk_kg', aggfunc='sum').fillna(0)

# Farm-level aggregate
farm_level = df.pivot_table(index='dtt', columns='Farm_Code', values='milk_kg', aggfunc='sum').fillna(0)

### Step 2 — Use the hts or statsforecast library
#### The best Python option today is hierarchicalforecast from Nixtla:

In [ ]:


pip install hierarchicalforecast statsforecast
from hierarchicalforecast.core import HierarchicalReconciliation
from hierarchicalforecast.methods import BottomUp, MinTrace
from statsforecast import StatsForecast
from statsforecast.models import AutoETS

# Define hierarchy as a list-of-lists: [farm, animal, series_key]
# Each row of df needs all three levels filled
df['Farm'] = df['Farm_Code'].astype(str)
df['Animal'] = df['Animal_ID']
df['Parity'] = 'P' + df['parity'].astype(str)

# Build the Y_df in Nixtla format: unique_id, ds, y
Y_df = df[['series_key', 'dtt', 'milk_kg']].rename(columns={
    'series_key': 'unique_id', 'dtt': 'ds', 'milk_kg': 'y'
})

# Also add upper-level series manually
animal_df = df.groupby(['Animal_ID', 'dtt'])['milk_kg'].sum().reset_index()
animal_df['unique_id'] = animal_df['Animal_ID']
animal_df = animal_df.rename(columns={'dtt': 'ds', 'milk_kg': 'y'})

farm_df = df.groupby(['Farm_Code', 'dtt'])['milk_kg'].sum().reset_index()
farm_df['unique_id'] = farm_df['Farm_Code'].astype(str)
farm_df = farm_df.rename(columns={'dtt': 'ds', 'milk_kg': 'y'})

# Combine all levels
Y_df_all = pd.concat([
    farm_df[['unique_id','ds','y']],
    animal_df[['unique_id','ds','y']],
    Y_df
]).sort_values(['unique_id', 'ds'])

### Step 3 — Define the hierarchy tags

In [ ]:
# S_df: summing matrix (columns = bottom series, rows = all series)
# tags: dict mapping each level name to the list of series at that level

tags = {
    'Farm':   farm_df['unique_id'].unique().tolist(),
    'Animal': animal_df['unique_id'].unique().tolist(),
    'Parity': df['series_key'].unique().tolist()
}

### Step 4 — Fit base forecasters and reconcile

In [ ]:
from statsforecast import StatsForecast
from statsforecast.models import AutoETS

# Fit one model per bottom-level series
sf = StatsForecast(models=[AutoETS(season_length=12)], freq='MS', n_jobs=-1)
sf.fit(Y_df_all)
forecasts_df = sf.predict(h=6)  # 6 periods ahead

# Reconcile: MinTrace is the gold standard
hrec = HierarchicalReconciliation(reconcilers=[
    BottomUp(),
    MinTrace(method='mint_shrink')
])

reconciled = hrec.reconcile(
    Y_hat_df=forecasts_df,
    Y_df=Y_df_all,
    tags=tags
)



## Gemini Method
